# LESSON 5.2: Computing Sinograms
## Image Reconstruction from Projections

In this lesson:
- How to compute the Radon Transform (sinogram) step by step
- Discrete implementation of line integrals
- Using scikit-image's `radon` function
- Reading and interpreting sinograms
- Effect of number of projections on sinogram quality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, rescale
from skimage.data import shepp_logan_phantom
from skimage.draw import disk, ellipse

## 1. Computing a Projection: Step by Step

To compute a projection at angle $\theta$, we need to:

1. **Rotate** the image by angle $\theta$ (or equivalently, rotate the ray direction)
2. **Sum** pixel values along each column (line integral)
3. The result is one row of the sinogram

### Discrete Radon Transform

For a discrete image $f[m, n]$ of size $M \times N$:

$$g[k, \theta_i] = \sum_{\text{pixels on ray } k} f[m, n]$$

where ray $k$ is the $k$-th parallel ray at angle $\theta_i$.

In [ ]:
from scipy.ndimage import rotate

# Create a simple test image
size = 128
image = np.zeros((size, size))
rr, cc = disk((64, 64), 30)
image[rr, cc] = 0.5
rr, cc = disk((50, 50), 10)
image[rr, cc] = 1.0

# Manual computation of a single projection
def compute_projection_manual(img, angle_deg):
    """Compute a single projection by rotating and summing columns."""
    # Rotate the image
    rotated = rotate(img, angle_deg, reshape=False, order=1)
    # Sum along columns (vertical direction)
    projection = np.sum(rotated, axis=0)
    return rotated, projection

# Show projections at different angles
angles_demo = [0, 30, 60, 90]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for i, angle in enumerate(angles_demo):
    rotated, projection = compute_projection_manual(image, angle)
    
    # Original with angle annotation
    axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Original ($\\theta$ = {angle}°)', fontsize=11)
    
    # Rotated image
    axes[1, i].imshow(rotated, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Rotated by {angle}°', fontsize=11)
    
    # Projection (sum of columns)
    axes[2, i].plot(projection, 'b-', linewidth=2)
    axes[2, i].set_title(f'Projection (column sums)', fontsize=11)
    axes[2, i].set_xlabel('Detector position')
    axes[2, i].grid(True, alpha=0.3)

plt.suptitle('Step-by-Step Projection Computation: Rotate → Sum Columns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each projection is computed by rotating the image and summing along columns.")
print("This is equivalent to computing line integrals along parallel rays at angle theta.")

## 2. Building a Sinogram Manually

A sinogram is constructed by stacking all projections at different angles.

In [ ]:
def compute_sinogram_manual(img, angles):
    """Compute the sinogram (Radon Transform) by rotating and summing."""
    num_angles = len(angles)
    num_detectors = img.shape[1]
    sinogram = np.zeros((num_detectors, num_angles))
    
    for i, angle in enumerate(angles):
        rotated = rotate(img, angle, reshape=False, order=1)
        sinogram[:, i] = np.sum(rotated, axis=0)
    
    return sinogram

# Compute sinogram manually
angles = np.linspace(0, 180, 180, endpoint=False)
sinogram_manual = compute_sinogram_manual(image, angles)

# Compare with scikit-image's radon function
sinogram_skimage = radon(image, theta=angles, circle=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original Image', fontsize=12)

axes[1].imshow(sinogram_manual, cmap='hot', aspect='auto',
              extent=[0, 180, sinogram_manual.shape[0], 0])
axes[1].set_title('Manual Sinogram', fontsize=12)
axes[1].set_xlabel('$\\theta$ (degrees)')
axes[1].set_ylabel('Detector position')

axes[2].imshow(sinogram_skimage, cmap='hot', aspect='auto',
              extent=[0, 180, sinogram_skimage.shape[0], 0])
axes[2].set_title('scikit-image Sinogram', fontsize=12)
axes[2].set_xlabel('$\\theta$ (degrees)')
axes[2].set_ylabel('Detector position')

plt.suptitle('Building a Sinogram: Manual vs scikit-image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Manual sinogram shape: {sinogram_manual.shape}")
print(f"scikit-image sinogram shape: {sinogram_skimage.shape}")
print("\nNote: scikit-image pads the image to ensure the entire circle is captured.")

## 3. Reading a Sinogram

### How to interpret a sinogram:

| Sinogram Feature | Meaning |
|---|---|
| Horizontal axis ($\theta$) | Projection angle |
| Vertical axis ($\rho$) | Detector position (distance from center) |
| Pixel intensity | Total attenuation along that ray |
| Sinusoidal curve | A point source in the original image |
| Bright region | Dense material (high attenuation) |
| Constant row | Rotationally symmetric feature at center |

In [ ]:
# Detailed sinogram analysis
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)  # reduce size for speed

theta = np.linspace(0., 180., 360, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Full sinogram
axes[0, 0].imshow(phantom, cmap='gray')
axes[0, 0].set_title('Shepp-Logan Phantom', fontsize=12)

im = axes[0, 1].imshow(sinogram, cmap='hot', aspect='auto',
                       extent=[0, 180, sinogram.shape[0], 0])
axes[0, 1].set_title('Complete Sinogram', fontsize=12)
axes[0, 1].set_xlabel('$\\theta$ (degrees)')
axes[0, 1].set_ylabel('$\\rho$')
plt.colorbar(im, ax=axes[0, 1])

# Individual projections
selected_angles = [0, 45, 90, 135]
colors = ['blue', 'red', 'green', 'purple']

for angle, color in zip(selected_angles, colors):
    idx = np.argmin(np.abs(theta - angle))
    axes[1, 0].plot(sinogram[:, idx], color=color, linewidth=1.5,
                   label=f'$\\theta$ = {angle}°')

axes[1, 0].set_title('Individual Projections', fontsize=12)
axes[1, 0].set_xlabel('Detector Position $\\rho$')
axes[1, 0].set_ylabel('Projection Value')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Show angular variation at a fixed detector position
center_idx = sinogram.shape[0] // 2
positions = [center_idx - 30, center_idx, center_idx + 30]
labels = ['$\\rho$ = center - 30', '$\\rho$ = center', '$\\rho$ = center + 30']

for pos, label in zip(positions, labels):
    axes[1, 1].plot(theta, sinogram[pos, :], linewidth=1.5, label=label)

axes[1, 1].set_title('Projection Values vs Angle (Fixed $\\rho$)', fontsize=12)
axes[1, 1].set_xlabel('$\\theta$ (degrees)')
axes[1, 1].set_ylabel('Projection Value')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Interpreting a Sinogram', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Effect of Number of Projections

The quality of reconstruction depends on how many projections (angles) are available.

In practice, the number of projections needed depends on:
- Image size ($N \times N$)
- Required resolution
- Noise level

A common rule of thumb: for an $N \times N$ image, at least $N$ equally-spaced projections are needed for a good reconstruction.

In [ ]:
# Effect of number of projection angles
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

num_projections = [10, 30, 90, 180, 360]

fig, axes = plt.subplots(2, len(num_projections), figsize=(20, 8))

for i, n_proj in enumerate(num_projections):
    theta = np.linspace(0., 180., n_proj, endpoint=False)
    sinogram = radon(phantom, theta=theta, circle=True)
    
    # Show sinogram
    axes[0, i].imshow(sinogram, cmap='hot', aspect='auto',
                     extent=[0, 180, sinogram.shape[0], 0])
    axes[0, i].set_title(f'{n_proj} projections', fontsize=11)
    axes[0, i].set_xlabel('$\\theta$')
    if i == 0:
        axes[0, i].set_ylabel('$\\rho$')
    
    # Show projection at theta=0
    axes[1, i].plot(sinogram[:, 0], 'b-', linewidth=1.5)
    axes[1, i].set_title(f'Projection at 0° ({n_proj} angles)', fontsize=10)
    axes[1, i].set_xlabel('Detector pos.')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Sinograms with Different Numbers of Projections', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("More projections → more information → better reconstruction potential.")
print("Too few projections → angular undersampling → artifacts in reconstruction.")

## 5. Limited-Angle and Sparse-Angle Sinograms

In practice, we may not always have full 180° coverage:

| Scenario | Description |
|----------|-------------|
| **Full-angle** | Projections from 0° to 180° (standard CT) |
| **Limited-angle** | Projections over a restricted angular range (e.g., 0° to 120°) |
| **Sparse-angle** | Few projections spread over 180° |

Both limited-angle and sparse-angle scenarios result in **incomplete sinograms** and produce artifacts in reconstruction.

In [ ]:
# Compare full, limited-angle, and sparse-angle sinograms
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

# Full angle (0-180, 180 projections)
theta_full = np.linspace(0, 180, 180, endpoint=False)
sinogram_full = radon(phantom, theta=theta_full, circle=True)

# Limited angle (0-120, 120 projections)
theta_limited = np.linspace(0, 120, 120, endpoint=False)
sinogram_limited = radon(phantom, theta=theta_limited, circle=True)

# Sparse angle (0-180, only 30 projections)
theta_sparse = np.linspace(0, 180, 30, endpoint=False)
sinogram_sparse = radon(phantom, theta=theta_sparse, circle=True)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Original', fontsize=12)

axes[1].imshow(sinogram_full, cmap='hot', aspect='auto',
              extent=[0, 180, sinogram_full.shape[0], 0])
axes[1].set_title(f'Full Angle\n(180 proj, 0°-180°)', fontsize=11)
axes[1].set_xlabel('$\\theta$')
axes[1].set_ylabel('$\\rho$')

axes[2].imshow(sinogram_limited, cmap='hot', aspect='auto',
              extent=[0, 120, sinogram_limited.shape[0], 0])
axes[2].set_title(f'Limited Angle\n(120 proj, 0°-120°)', fontsize=11)
axes[2].set_xlabel('$\\theta$')

axes[3].imshow(sinogram_sparse, cmap='hot', aspect='auto',
              extent=[0, 180, sinogram_sparse.shape[0], 0])
axes[3].set_title(f'Sparse Angle\n(30 proj, 0°-180°)', fontsize=11)
axes[3].set_xlabel('$\\theta$')

plt.suptitle('Full vs Limited vs Sparse-Angle Sinograms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Full angle: Complete information for reconstruction.")
print("Limited angle: Missing angular range causes elongation artifacts.")
print("Sparse angle: Angular undersampling causes streak artifacts.")

## 6. Sinogram of Biomedical Images

Let's compute sinograms of more realistic phantom images.

In [ ]:
# Create a more complex biomedical phantom
size = 256
phantom_bio = np.zeros((size, size))

# Outer body (skull)
rr, cc = ellipse(128, 128, 110, 90)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 0.3

# Brain tissue
rr, cc = ellipse(128, 128, 100, 80)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 0.5

# Ventricles
rr, cc = ellipse(115, 110, 20, 10)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 0.2

rr, cc = ellipse(115, 146, 20, 10)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 0.2

# Tumor (small, high density)
rr, cc = disk((150, 160), 12)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 0.9

# Calcification
rr, cc = disk((100, 90), 5)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_bio[rr[valid], cc[valid]] = 1.0

# Compute sinogram
theta = np.linspace(0., 180., 360, endpoint=False)
sinogram_bio = radon(phantom_bio, theta=theta, circle=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(phantom_bio, cmap='gray')
axes[0].set_title('Brain Phantom', fontsize=12)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(sinogram_bio, cmap='hot', aspect='auto',
                     extent=[0, 180, sinogram_bio.shape[0], 0])
axes[1].set_title('Sinogram', fontsize=12)
axes[1].set_xlabel('$\\theta$ (degrees)')
axes[1].set_ylabel('$\\rho$')
plt.colorbar(im1, ax=axes[1])

# Show 3D surface of sinogram
ax3d = fig.add_subplot(1, 3, 3, projection='3d')
T, R = np.meshgrid(theta[::4], np.arange(sinogram_bio.shape[0])[::4])
ax3d.plot_surface(T, R, sinogram_bio[::4, ::4], cmap='hot', alpha=0.8)
ax3d.set_xlabel('$\\theta$')
ax3d.set_ylabel('$\\rho$')
ax3d.set_zlabel('Value')
ax3d.set_title('Sinogram (3D View)', fontsize=12)

plt.suptitle('Biomedical Phantom and Its Sinogram', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Noise in Sinograms

In real CT scanning, projections are corrupted by noise:
- **Photon counting noise** (Poisson statistics)
- **Electronic noise** (detector electronics)
- **Scatter radiation**

Noise in the sinogram leads to artifacts in the reconstructed image.

In [ ]:
# Effect of noise on sinograms
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

# Add different levels of Gaussian noise
noise_levels = [0.0, 0.5, 2.0, 5.0]

fig, axes = plt.subplots(2, len(noise_levels), figsize=(18, 8))

for i, noise_std in enumerate(noise_levels):
    if noise_std == 0:
        sinogram_noisy = sinogram_clean.copy()
    else:
        noise = np.random.normal(0, noise_std, sinogram_clean.shape)
        sinogram_noisy = sinogram_clean + noise
    
    # Show noisy sinogram
    axes[0, i].imshow(sinogram_noisy, cmap='hot', aspect='auto',
                     extent=[0, 180, sinogram_noisy.shape[0], 0])
    axes[0, i].set_title(f'Noise σ = {noise_std}', fontsize=11)
    axes[0, i].set_xlabel('$\\theta$')
    if i == 0:
        axes[0, i].set_ylabel('$\\rho$')
    
    # Show a single projection comparison
    axes[1, i].plot(sinogram_clean[:, 0], 'b-', linewidth=1.5, label='Clean', alpha=0.7)
    axes[1, i].plot(sinogram_noisy[:, 0], 'r-', linewidth=1, label='Noisy', alpha=0.7)
    axes[1, i].set_title(f'Projection at 0° (σ={noise_std})', fontsize=10)
    axes[1, i].set_xlabel('Detector pos.')
    axes[1, i].legend(fontsize=8)
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Effect of Noise on Sinograms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Noise in projections will propagate and amplify during reconstruction.")
print("This is why noise handling is critical in CT reconstruction algorithms.")

## Summary

What we learned:
1. A projection is computed by **rotating the image** and **summing along columns** (line integrals)
2. The **sinogram** is built by stacking projections from all angles
3. scikit-image's `radon()` function efficiently computes the Radon Transform
4. Sinograms encode spatial information: a point source appears as a **sinusoidal curve**
5. More projection angles → more information → better reconstruction quality
6. **Limited-angle** and **sparse-angle** scenarios lead to incomplete sinograms and reconstruction artifacts
7. **Noise** in projections corrupts the sinogram and amplifies during reconstruction